# Packages

## The Problem

As a project grows, the number of Python modules grows too.

Keeping dozens of related `.py` files in the same directory makes the project harder to organize.

## From Modules to Packages

A module is usually a `.py` file containing reusable Python objects.

For example:

```text
project/
├── arithmetic.py
├── geometry.py
└── statistics.py
````

These modules are useful, but they are all in the same directory.

## What Is a Package?

A package is a directory that organizes related modules.

In [1]:
from pathlib import Path

package = Path("package_calculations")

package.mkdir(exist_ok=True)

(package / "arithmetic.py").write_text(
    "def add(a, b):\n"
    "    return a + b\n"
)

(package / "geometry.py").write_text(
    "def area(side):\n"
    "    return side ** 2\n"
)

print(package.is_dir())
print((package / "arithmetic.py").is_file())
print((package / "geometry.py").is_file())

True
True
True


Our project now looks like this:

```text
package_calculations/
├── arithmetic.py
└── geometry.py
```

The directory groups the modules together.

The basic relationship is:

```text
package → modules
```

In our example:

`package_calculations` → package

`arithmetic.py` → module

`geometry.py` → module

## Importing the Package

Now import only the package.

The package contains `arithmetic.py` and `geometry.py`, but these modules are not automatically imported.

In [2]:
import package_calculations

print(hasattr(package_calculations, "arithmetic"))
print(hasattr(package_calculations, "geometry"))

False
False


Both results are `False`.

The package exists, but its modules were not automatically imported.

This is why packages and their modules are separate concepts.

## Importing a Module from a Package

A module can be imported through its package path.

In [3]:
import package_calculations.arithmetic

result = package_calculations.arithmetic.add(10, 20)

print(result)

30


The dot connects the package and the module:

`package_calculations.arithmetic`

The module `arithmetic` belongs to the package `package_calculations`.

## `__init__.py`

A package commonly contains a special file called `__init__.py`.

It can be empty or contain Python code.

Let's add it to our package.

In [4]:
from pathlib import Path

package = Path("package_calculations")

(package / "__init__.py").write_text("")

print((package / "__init__.py").is_file())

True


The package now has:

```text
package_calculations/
├── __init__.py
├── arithmetic.py
└── geometry.py
```

`__init__.py` belongs to the package itself.

## Exposing an Object

`__init__.py` can import objects from the package's modules.

Let's expose `add` from `arithmetic.py`.

In [5]:
from pathlib import Path

package = Path("package_calculations")

(package / "__init__.py").write_text(
    "from .arithmetic import add\n"
)

print((package / "__init__.py").read_text())

from .arithmetic import add



The import:

`from .arithmetic import add`

means:

import `add` from `arithmetic` inside the current package.

The `.` indicates the current package.

In [6]:
import importlib
import package_calculations

importlib.reload(package_calculations)

print(package_calculations.add(10, 20))

30


`add` is now available directly from the package.

Instead of:

`package_calculations.arithmetic.add`

we can use:

`package_calculations.add`

The function is still defined in `arithmetic.py`.

## Package Interface

The package can expose selected objects through `__init__.py`.

This creates a simpler interface:

package_calculations.add

while the internal structure remains:

```text
package_calculations/
├── __init__.py
├── arithmetic.py
└── geometry.py
```

In [7]:
import package_calculations

print(hasattr(package_calculations, "add"))
print(hasattr(package_calculations, "area"))

True
False


`add` is available because `__init__.py` exposes it.

`area` is not exposed.

The package can therefore provide a simple interface without exposing every object from every module.

## Relative Imports

Modules inside the same package can import from each other.

For example, `geometry.py` can use `add` from `arithmetic.py` with:

`from .arithmetic import add`

The `.` means:

**from the current package.**

In [8]:
from pathlib import Path

package = Path("package_calculations")

(package / "geometry.py").write_text(
    "from .arithmetic import add\n\n"
    "def double(value):\n"
    "    return add(value, value)\n"
)

print((package / "geometry.py").read_text())

from .arithmetic import add

def double(value):
    return add(value, value)



Now the two modules are connected:

```text
package_calculations/
├── arithmetic.py
└── geometry.py
````

`geometry.py` imports `add` from `arithmetic.py`.

The package provides the common structure for both modules.

In [9]:
import importlib
import package_calculations.geometry

importlib.reload(package_calculations.geometry)

result = package_calculations.geometry.double(10)

print(result)

20


# Summary

- A package organizes related modules.
- A module is usually a `.py` file.
- A package can contain multiple modules.
- Importing a package does not automatically import all its modules.
- `__init__.py` belongs to the package.
- `__init__.py` can expose selected objects from modules.
- Relative imports use `.` to refer to the current package.
- The basic hierarchy is package → module → object.